# f696/b472 开发期现金转 SOXL 的合法目标支持度

只读取封存的 Qlib Recorder 与决策前状态。先核对来源身份、NAS 已验证收据、1756 行提案与投影；随后按冻结的 +0.10 beta 诊断上界统计跨年支持，不读取未来收益或拟合。

In [1]:
import hashlib, json, os, pathlib, subprocess, sys, tempfile
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import numpy as np, pandas as pd, qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1]
sys.path.insert(0,str(root))
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.data.nas_market_data import archive_nas_recorder, materialize_nas_evidence, _settings
card=root/'research/stage1_f696_b472_soxl_training_support_20261025'
plan=json.loads((card/'plan.json').read_text()); src=plan['sources']
sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'result.json').exists()
prior=root/src['prior_lineage_card']
assert sha(prior/'summary.json')==src['prior_lineage_summary_sha256']
prior_summary=json.loads((prior/'summary.json').read_text())
assert prior_summary['source_legacy_full_Recorder_NAS_scoped_restore_verified'] and prior_summary['NAS_readback']['verified']
a_card=root/src['allocation_card']; b_card=root/src['action_card']
assert sha(a_card/'summary.json')==src['allocation_summary_sha256']
assert sha(b_card/'summary.json')==src['action_summary_sha256']
assert sha(root/'src/workflow/intraday_recovery_direct_allocation.py')==src['projector_module_sha256']
a_rel='artifacts/pit_constituent_auxiliary_training_allocation_retrained_20260928/predictions.csv'
b_rel='artifacts/stage1_open_action_value_net_fees_20260928/action_value_training_frame.csv'
a_path=a_card/'qlib_recorder'/src['allocation_experiment_id']/src['allocation_recorder_id']/a_rel
b_path=b_card/'qlib_recorder'/src['action_experiment_id']/src['action_recorder_id']/b_rel
for path, expected in [(a_path,src['allocation_predictions_artifact_sha256']),(b_path,src['action_training_frame_artifact_sha256'])]:
    assert path.is_file() and path.stat().st_size>0 and sha(path)==expected
    assert subprocess.run(['git','ls-files','--error-unmatch',str(path.relative_to(root))],cwd=root,capture_output=True).returncode==0
provider=(root/src['Qlib_provider_uri']).resolve()
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str((a_card/'qlib_recorder').resolve()),'default_exp_name':a_card.name}})
with tempfile.TemporaryDirectory(prefix='stage1-f696-soxl-support-input-') as tmp:
    a_rec=R.get_recorder(experiment_id=src['allocation_experiment_id'],recorder_id=src['allocation_recorder_id'])
    assert a_rec.status=='FINISHED'
    a_dl=pathlib.Path(a_rec.download_artifact('pit_constituent_auxiliary_training_allocation_retrained_20260928/predictions.csv',tmp))
    assert sha(a_dl)==sha(a_path)
    with R.uri_context(str((b_card/'qlib_recorder').resolve())):
        b_rec=R.get_recorder(experiment_id=src['action_experiment_id'],recorder_id=src['action_recorder_id'])
        assert b_rec.status=='FINISHED'
        b_dl=pathlib.Path(b_rec.download_artifact('stage1_open_action_value_net_fees_20260928/action_value_training_frame.csv',tmp))
        assert sha(b_dl)==sha(b_path)
        a=pd.read_csv(a_dl,usecols=['signal_date','segment','soxl_weight','soxx_weight','cash_weight'])
        b=pd.read_csv(b_dl,usecols=['signal_date','year','raw_soxl_weight','raw_soxx_weight','raw_cash_weight','current_beta_cap','target_soxl_weight','target_soxx_weight','target_cash_weight','decision_at'])
a=a[a.segment.eq('development_walk_forward')].copy()
assert len(a)==len(b)==1756 and a.signal_date.is_unique and b.signal_date.is_unique
m=a.merge(b,on='signal_date',validate='one_to_one',how='outer',indicator=True)
assert len(m)==1756 and m._merge.eq('both').all() and m.signal_date.is_monotonic_increasing
assert (pd.to_datetime(m.decision_at,utc=True)>pd.to_datetime(m.signal_date,utc=True)).all()
for x,y in [('soxl_weight','raw_soxl_weight'),('soxx_weight','raw_soxx_weight'),('cash_weight','raw_cash_weight')]:
    assert np.max(np.abs(m[x]-m[y]))==0
calendar={str(x)[:10] for x in D.calendar(start_time='2016-01-04',end_time='2022-12-21',freq='day')}
assert set(m.signal_date).issubset(calendar)
rows=[];maxerr=0.0
for x in m.itertuples(index=False):
    sx=float(x.raw_soxl_weight);xx=float(x.raw_soxx_weight);cash=float(x.raw_cash_weight);cap=float(x.current_beta_cap)
    saved=np.array([float(x.target_soxl_weight),float(x.target_soxx_weight),float(x.target_cash_weight)])
    assert np.isfinite([sx,xx,cash,cap]).all() and min(sx,xx,cash)>=-1e-9 and abs(sx+xx+cash-1)<1e-7
    control=project_legal_weights(sx,xx,cap,minimum_cash_weight=.15,maximum_soxl_weight=.30)
    err=float(np.max(np.abs(control-saved)));maxerr=max(maxerr,err);assert err<1e-8,(x.signal_date,err)
    rawbeta=3*sx+xx;basebeta=3*control[0]+control[1]
    binding=bool(rawbeta>cap+1e-9);ceiling=cap+.10 if binding else cap
    delta=max(0.,min(.10,max(0.,ceiling-basebeta),3*max(0.,control[2]-.15),3*max(0.,.30-control[0])))
    candidate=control+np.array([delta/3,0.,-delta/3])
    assert abs(candidate.sum()-1)<1e-9 and candidate[0]<=.30+1e-9 and candidate[2]>=.15-1e-9
    assert 3*candidate[0]+candidate[1]<=ceiling+1e-8
    rows.append({'signal_date':x.signal_date,'year':int(x.year),'binding':binding,'raw_cash_weight':cash,'raw_beta':rawbeta,'current_beta_cap':cap,'projected_beta':basebeta,'legal_delta_beta':delta,'positive_support':bool(delta>1e-10)})
table=pd.DataFrame(rows)
annual=[{'year':int(y),'rows':len(g),'binding_rows':int(g.binding.sum()),'positive_support':int(g.positive_support.sum()),'binding_positive':int((g.binding & g.positive_support).sum()),'nonbinding_positive':int((~g.binding & g.positive_support).sum())} for y,g in table.groupby('year')]
n=int(table.positive_support.sum());years20=sum(z['positive_support']>=20 for z in annual);gate=bool(n>=100 and years20>=3)
result={'decision':'source_support_pass_separate_information_gate_required' if gate else 'fixed_SOXL_training_target_support_failed','plan_sha256':sha(card/'plan.json'),'source_recorder_ids':[a_rec.id,b_rec.id],'sessions':len(table),'exact_raw_allocation_match_rows':len(table),'max_saved_target_projection_error':maxerr,'binding_rows':int(table.binding.sum()),'positive_support':n,'binding_positive':int((table.binding & table.positive_support).sum()),'nonbinding_positive':int((~table.binding & table.positive_support).sum()),'years_with_at_least20':years20,'support_gate_pass':gate,'annual':annual,'future_label_columns_loaded':0,'model_fits':0,'native_accounts':0,'scope':'legal target support only; no OOF b472 action, physical fill, local net-value label or own-state funded candidate account'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.uri_context(str(card/'qlib_recorder')):
    with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
        rec=R.get_recorder()
        rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'allocation_predictions':sha(a_path),'action_training_frame':sha(b_path)})
        rec.save_objects(artifact_path='analysis',source_support=table,result=result)
        rec.log_metrics(source_rows=len(table),positive_support=n,binding_positive=result['binding_positive'],nonbinding_positive=result['nonbinding_positive'],support_gate_pass=int(gate),model_fits=0,native_accounts=0)
        rid,eid=rec.id,str(rec.experiment_id)
binding=card/'nas_binding.json'
binding.write_text(json.dumps({'schema_version':'1','authority':'yanxuan_nas','source_id':plan['experiment'],'as_of':'2026-10-25','source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='stage1-f696-soxl-support-restore-') as tmp:
    temporary=pathlib.Path(tmp);manifest=temporary/'nas_manifest.json'
    manifest.write_text(json.dumps({'evidence':evidence,'ssh':_settings(root)['ssh']}))
    restored=temporary/'card';restored.mkdir()
    materialize_nas_evidence(manifest,card=restored)
    with R.uri_context(str(restored/'qlib_recorder')):
        cold=R.get_recorder(experiment_id=eid,recorder_id=rid)
        assert cold.status=='FINISHED'
        cold_result=cold.load_object('analysis/result')
        assert cold_result==result
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_ids':result['source_recorder_ids'],'source_recorder_NAS_scoped_restore_verified':True,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'verified':True,'archive_sha256':evidence['sha256'],'selected_result_equal':True},'positive_support':n,'support_gate_pass':gate,'future_label_columns_loaded':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/source_support','analysis/result']},'limitations':['No OOF b472 action decision, whole-share fill, local fee-net value or funded SOXX account.','Cap+0.10 on binding dates is a diagnostic legal-target envelope, not an approved policy.']}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':result['decision'],'support':n,'binding_positive':result['binding_positive'],'nonbinding_positive':result['nonbinding_positive'],'annual':annual,'recorder_id':rid},ensure_ascii=False))

{"decision": "source_support_pass_separate_information_gate_required", "support": 911, "binding_positive": 859, "nonbinding_positive": 52, "annual": [{"year": 2016, "rows": 252, "binding_rows": 52, "positive_support": 53, "binding_positive": 52, "nonbinding_positive": 1}, {"year": 2017, "rows": 251, "binding_rows": 32, "positive_support": 67, "binding_positive": 32, "nonbinding_positive": 35}, {"year": 2018, "rows": 251, "binding_rows": 126, "positive_support": 140, "binding_positive": 126, "nonbinding_positive": 14}, {"year": 2019, "rows": 252, "binding_rows": 236, "positive_support": 236, "binding_positive": 236, "nonbinding_positive": 0}, {"year": 2020, "rows": 253, "binding_rows": 164, "positive_support": 166, "binding_positive": 164, "nonbinding_positive": 2}, {"year": 2021, "rows": 252, "binding_rows": 35, "positive_support": 35, "binding_positive": 35, "nonbinding_positive": 0}, {"year": 2022, "rows": 245, "binding_rows": 214, "positive_support": 214, "binding_positive": 214, "n

# f696/b472 开发期现金转 SOXL：合法目标支持度通过

冻结读取 f696 的 2016–2022 `development_walk_forward` 配置预测与 b472 同日期决策前训练状态。两份 Recorder 的原始三资产权重 **1756/1756 行完全一致**，原风险投影复现保存目标的最大误差 **2.22e-16**；Qlib 日历、来源 SHA、既有来源 NAS 限定回读均核对。未读取未来标签、拟合模型或运行账户。

诊断动作仅在原合法投影后从现金转最多 **0.10 beta** 给 SOXL，仍守 SOXL≤30%、现金≥15%。原始 beta 超过自身上限的日期，诊断上界设为原上限+0.10；其他日期保持原上限。这是**目标空间上界，不是批准的风险策略或资金反事实**。1756 日中有 **911 日**合法目标增配量大于零，七个年份各至少 20 日，超过冻结的总数100且至少三年各20日的来源支持门。859/911 日在受限层；非受限仅52日，其中2019、2021、2022均为0。各年及受限/非受限计数见 `result.json`，日级表保存在原生 Qlib Recorder。

决定 `source_support_pass_separate_information_gate_required`：受限层有跨年训练支持，可以继续冻结相应的费用后局部价值标签与事前信息门；但这一动作几乎不触及开发期非受限现金层，不能单靠它声称能修复完整126日稳定性。b472训练框架不是逐年向前的实际动作账户，本卡也未验证整股成交、费用后局部收益、候选自身风险反馈或 SOXX 净超额。下一步先确认能否用封存的 f696/b472 开发期资料构造**同状态、合法整股、0bp仍扣Tiered费、共同次日复位**的局部训练标签；如缺少物理账户状态，应明确补建训练参考并冻结预算，不能把原 `execute_advantage` 当作本动作标签。

执行notebook、来源原生Recorder及NAS归档均完成；新Recorder `8d0400b6bfdd46b49694745b1254d0ee`，NAS归档 SHA-256 `69e9b58ea188f3b615e2fb6437dcfd5667f8e67c877eadcff47fb0ce8703b4d4`、大小 **98,837 字节**，限定冷恢复并用原生 Qlib 重读 `analysis/result` 与本卡逐字典一致。SOXX历史0/2/5bp门均未在本卡测试，Goal继续 active。
